In [0]:
from pyspark.sql import functions
from pyspark.sql.types import DecimalType

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_movies")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlproyecto12")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/MoreInfo.csv"

In [0]:
df_moreinfo = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
moreinfo_schema = StructType(fields=[StructField("id", IntegerType(), False),
                                     StructField("runtime", StringType(), True),
                                     StructField("budget", StringType(), True),
                                     StructField("revenue", StringType(), True),
                                     StructField("film_id", IntegerType(), True)
])

In [0]:
df_moreinfo_final = spark.read\
.option('header', True)\
.schema(moreinfo_schema)\
.csv(ruta)

In [0]:
moreinfo_selected_df = df_moreinfo_final.select(col("id"), 
                                                col("runtime"), 
                                                col("budget"),
                                                col("revenue")
                                                )

In [0]:
moreinfo_with_timestamp_df = moreinfo_selected_df.withColumn("ingestion_date", current_timestamp())

In [0]:
display(moreinfo_with_timestamp_df.limit(5))

In [0]:
moreinfo_with_timestamp_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.moviemoreinfo")

In [0]:
%sql
select count(1) from catalog_movies.bronze.moviemoreinfo